# 01_Config path and caese

In [16]:
import json
import math
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import pandas as pd
from IPython.display import display

In [17]:
SHARED_DIR = Path("/workspace/shared")

if not SHARED_DIR.is_dir():
    SHARED_DIR = Path.cwd() 

assert SHARED_DIR.is_dir(), f"ไม่พบ shared directory: {SHARED_DIR}"

In [18]:
RESULTS_DIR = SHARED_DIR / "results"
SUMMARY_DIR = SHARED_DIR / "summaries" 
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

SDK_ORDER = [
    "deepagents",
    "openai_agents",
    "pi_agent",
]

SDK_LABELS = {
    "deepagents": "Deep Agents SDK",
    "openai_agents": "OpenAI Agents SDK",
    "pi_agent": "Pi Agent SDK",
}

EXPECTED_CASES = [
    "math_001",
    "math_002",
    "math_003",
    "mcp_001",
    "mcp_002",
    "mcp_003",
    "web_native_001",
    "web_native_002",
    "planning_001",
    "planning_002",
    "long_context_001",
    "long_context_002",
    "history_001",
    "history_002",
    "history_003",
    "error_001",
    "error_002",
]

SELECTION_POLICY = "latest_run_per_sdk_and_case"

print("Results:", RESULTS_DIR)
print("Summary:", SUMMARY_DIR)

Results: /Users/tun.k/Desktop/PTT_project/Agent_SDKs/shared/results
Summary: /Users/tun.k/Desktop/PTT_project/Agent_SDKs/shared/summaries


# 02 Read through whole the files

In [19]:
def sdk_from_path(path):
    parts = [part.lower() for part in path.parts]

    if any(part.startswith("deepagents") for part in parts):
        return "deepagents"

    if any(
        part.startswith("openai_agents")
        or part.startswith("openai-agent")
        for part in parts
    ):
        return "openai_agents"

    if any(
        part in {"pi_agent", "pi_agents"}
        for part in parts
    ):
        return "pi_agent"

    return None


def add_payload_records(payload, source_path, records):
    if isinstance(payload, dict) and isinstance(
        payload.get("results"), list
    ):
        for index, record in enumerate(payload["results"]):
            if isinstance(record, dict):
                item = dict(record)
                item["_source_file"] = str(source_path)
                item["_source_index"] = index
                records.append(item)

    elif isinstance(payload, dict) and payload.get("case_id"):
        item = dict(payload)
        item["_source_file"] = str(source_path)
        item["_source_index"] = 0
        records.append(item)


raw_records = []
read_errors = []

for path in sorted(RESULTS_DIR.rglob("*.json")):
    try:
        payload = json.loads(
            path.read_text(encoding="utf-8")
        )
        add_payload_records(payload, path, raw_records)
    except Exception as exc:
        read_errors.append({
            "file": str(path),
            "error": f"{type(exc).__name__}: {exc}",
        })

for record in raw_records:
    record["_sdk_folder"] = sdk_from_path(
        Path(record["_source_file"])
    )

records = [
    record
    for record in raw_records
    if record["_sdk_folder"] in SDK_ORDER
]

print("JSON files read:", len(list(RESULTS_DIR.rglob("*.json"))))
print("Records loaded:", len(records))
print("Read errors:", len(read_errors))

if read_errors:
    display(pd.DataFrame(read_errors))

JSON files read: 55
Records loaded: 72
Read errors: 0


# 03 Remove duplicate file (Select only lasted file)

In [20]:
def parse_datetime(value):
    if not isinstance(value, str):
        return datetime.min.replace(tzinfo=timezone.utc)

    try:
        parsed = datetime.fromisoformat(
            value.replace("Z", "+00:00")
        )

        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)

        return parsed.astimezone(timezone.utc)

    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


def record_time(record):
    for field in [
        "started_at_utc",
        "started_at",
        "saved_at_utc",
        "exported_at_utc",
    ]:
        parsed = parse_datetime(record.get(field))

        if parsed != datetime.min.replace(tzinfo=timezone.utc):
            return parsed

    source_path = Path(record["_source_file"])

    try:
        return datetime.fromtimestamp(
            source_path.stat().st_mtime,
            tz=timezone.utc,
        )
    except OSError:
        return datetime.min.replace(tzinfo=timezone.utc)


# ขั้นแรก: ตัด duplicate export ของ run_id เดียวกัน
unique_runs = {}

for record in records:
    sdk = record["_sdk_folder"]
    case_id = record.get("case_id", "unknown")
    run_id = record.get("run_id")

    if not run_id:
        run_id = (
            f"file:{record['_source_file']}"
            f"#index:{record['_source_index']}"
        )

    key = (sdk, case_id, run_id)

    current = unique_runs.get(key)

    if current is None or record_time(record) >= record_time(current):
        unique_runs[key] = record


# ขั้นที่สอง: เลือก Run ล่าสุดของแต่ละ SDK ต่อแต่ละ Case
selected_runs = {}

for record in unique_runs.values():
    key = (
        record["_sdk_folder"],
        record.get("case_id", "unknown"),
    )

    current = selected_runs.get(key)

    if current is None or record_time(record) >= record_time(current):
        selected_runs[key] = record


selected_records = list(selected_runs.values())

print("Records before deduplication:", len(records))
print("Unique runs:", len(unique_runs))
print("Selected latest runs:", len(selected_records))

Records before deduplication: 72
Unique runs: 69
Selected latest runs: 53


# 04 Normalize Results for any (SDK)

In [21]:
def to_bool_or_none(value):
    if type(value) is bool:
        return value

    return None


def get_task_passed(record):
    return to_bool_or_none(record.get("task_passed"))


def get_status(record):
    runtime_status = str(
        record.get("runtime_status", "")
    ).lower()

    task_passed = get_task_passed(record)

    explicit_status = record.get("evaluation_status")

    if explicit_status in {
        "unsupported",
        "runtime_error",
        "pending_review",
        "passed",
        "failed",
    }:
        return explicit_status

    if runtime_status not in {
        "",
        "completed",
        "success",
        "succeeded",
    }:
        return "runtime_error"

    if record.get("unsupported") is True:
        return "unsupported"

    if task_passed is True:
        return "passed"

    if task_passed is False:
        return "failed"

    return "pending_review"


def get_tool_call_count(record):
    numeric_fields = [
        "tool_call_count",
        "server_tool_calls",
        "mcp_calls",
        "observed_web_search_calls",
    ]

    for field in numeric_fields:
        value = record.get(field)

        if isinstance(value, (int, float)) and not isinstance(value, bool):
            return int(value)

    for field in [
        "tool_trace",
        "tool_calls",
        "server_trace",
        "web_action_trace",
    ]:
        value = record.get(field)

        if isinstance(value, list):
            return len(value)

    action_counts = record.get("action_counts")

    if isinstance(action_counts, dict):
        values = [
            value
            for value in action_counts.values()
            if isinstance(value, (int, float))
        ]

        if values:
            return int(sum(values))

    return None


def get_total_tokens(record):
    direct_value = record.get("total_tokens")

    if isinstance(direct_value, (int, float)):
        return int(direct_value)

    usage_values = (
        record.get("usage_per_visible_ai_message")
        or record.get("usage")
        or []
    )

    if isinstance(usage_values, dict):
        usage_values = [usage_values]

    totals = []

    for usage in usage_values:
        if isinstance(usage, dict):
            value = usage.get("total_tokens")

            if isinstance(value, (int, float)):
                totals.append(value)

    return int(sum(totals)) if totals else None


def get_checks_pass_rate(record):
    checks = record.get("checks")

    if not isinstance(checks, dict):
        return None

    boolean_values = [
        value
        for value in checks.values()
        if type(value) is bool
    ]

    if not boolean_values:
        return None

    return round(
        sum(boolean_values) / len(boolean_values),
        4,
    )


def get_category(case_id):
    if case_id.startswith("web_native"):
        return "web_search"

    return case_id.split("_")[0]


normalized_rows = []

for record in selected_records:
    sdk = record["_sdk_folder"]
    case_id = record.get("case_id", "unknown")

    normalized_rows.append({
        "sdk": sdk,
        "sdk_label": SDK_LABELS[sdk],
        "case_id": case_id,
        "category": get_category(case_id),
        "run_id": record.get("run_id"),
        "status": get_status(record),
        "task_passed": get_task_passed(record),
        "runtime_status": record.get("runtime_status"),
        "latency_s": record.get("latency_s"),
        "tool_call_count": get_tool_call_count(record),
        "total_tokens": get_total_tokens(record),
        "checks_pass_rate": get_checks_pass_rate(record),
        "source_file": record["_source_file"],
        "error": record.get("error"),
        "workflow_passed": record.get("workflow_passed"),
        "web_search_observed": record.get(
            "web_search_observed"
        ),
        "web_fetch_observed": record.get(
            "web_fetch_observed"
        ),
    })

normalized_df = pd.DataFrame(normalized_rows)

normalized_df["sdk_label"] = pd.Categorical(
    normalized_df["sdk_label"],
    categories=[
        SDK_LABELS[sdk]
        for sdk in SDK_ORDER
    ],
    ordered=True,
)

normalized_df = normalized_df.sort_values(
    ["case_id", "sdk_label"]
).reset_index(drop=True)

display(normalized_df)

,sdk,sdk_label,case_id,category,run_id,status,task_passed,runtime_status,latency_s,tool_call_count,total_tokens,checks_pass_rate,source_file,error,workflow_passed,web_search_observed,web_fetch_observed
0,deepagents,Deep Agents SDK,error_001,error,b5feace7c739496fa5fd6ca5f6856ca9,passed,True,completed,4.260000,1.0,4560.0,1.0000,/Users/tun.k/Desktop/PTT_project/Agent_SDKs/sh...,None,None,None,None
1,openai_agents,OpenAI Agents SDK,error_001,error,f5a2085614854170bfaab4978cd22ebc,passed,True,completed,4.325311,1.0,720.0,1.0000,/Users/tun.k/Desktop/PTT_project/Agent_SDKs/sh...,None,None,None,None
2,pi_agent,Pi Agent SDK,error_001,error,998887ccc0b44fbb86aca36457bbf1c9,passed,True,completed,NaN,1.0,708.0,1.0000,/Users/tun.k/Desktop/PTT_project/Agent_SDKs/sh...,None,None,None,None
3,deepagents,Deep Agents SDK,error_002,error,0852ec9a0baa485bab4d1d0ebadc1587,passed,True,completed,6.810000,2.0,7074.0,1.0000,/Users/tun.k/Desktop/PTT_project/Agent_SDKs/sh...,None,None,None,None
4,openai_agents,OpenAI Agents SDK,error_002,error,19e02babf7664d86952132eac5bb373f,passed,True,completed,4.806594,2.0,1321.0,1.0000,/Users/tun.k/Desktop/PTT_project/Agent_SDKs/sh...,None,None,None,None
5,pi_agent,Pi Agent SDK,error_002,error,5817997a2d144bb08162ff714228e6f9,passed,True,completed,NaN,2.0,1323.0,1.0000,/Users/tun.k/Desktop/PTT_project/Agent_SDKs/sh...,None,None,None,None
6,deepagents,Deep Agents SDK,history_001,history,afe0284ef1884bc19a2200695da6a03e,passed,True,completed,4.580000,NaN,4342.0,1.0000,/Users/tun.k/Desktop/PTT_project/Agent_SDKs/sh...,None,None,None,None
7,openai_agents,OpenAI Agents SDK,history_001,history,e952cf7d1a254cdd9ce27772f4a44356,passed,True,completed,4.555078,0.0,420.0,1.0000,/Users/tun.k/Desktop/PTT_project/Agent_SDKs/sh...,None,None,None,None
8,pi_agent,Pi Agent SDK,history_001,history,ae4f2bf03ba84156be37b17580f5f7d3,passed,True,completed,NaN,0.0,499.0,1.0000,/Users/tun.k/Desktop/PTT_project/Agent_SDKs/sh...,None,None,None,None
9,deepagents,Deep Agents SDK,history_002,history,8436c94e1bde40799643ec2ec1b007d8,passed,True,completed,8.810000,0.0,4882.0,1.0000,/Users/tun.k/Desktop/PTT_project/Agent_SDKs/sh...,None,None,None,None


# 05 Table to compare each caese

In [22]:
STATUS_LABELS = {
    "passed": "PASS",
    "failed": "FAIL",
    "pending_review": "PENDING",
    "runtime_error": "RUNTIME_ERROR",
    "unsupported": "UNSUPPORTED",
}

status_df = normalized_df.copy()

status_df["result"] = status_df["status"].map(
    STATUS_LABELS
)

status_matrix = (
    status_df
    .pivot_table(
        index="case_id",
        columns="sdk_label",
        values="result",
        aggfunc="first",
    )
    .reindex(columns=[
        SDK_LABELS[sdk]
        for sdk in SDK_ORDER
    ])
)

display(status_matrix)

/var/folders/mr/ldw7p0vs64x3yhr06hnqx3qm0000gp/T/ipykernel_27665/4250618013.py:17: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


sdk_label,Deep Agents SDK,OpenAI Agents SDK,Pi Agent SDK
case_id,,,
error_001,PASS,PASS,PASS
error_002,PASS,PASS,PASS
history_001,PASS,PASS,PASS
history_002,PASS,PASS,PASS
history_003,PASS,PASS,PASS
long_context_001,PASS,PASS,PASS
long_context_002,PASS,PASS,PASS
math_001,PASS,PASS,PASS
math_002,PASS,PASS,PASS


# 6 Summarize Socre for each SDK

In [23]:
def median_or_none(series):
    values = [
        float(value)
        for value in series
        if isinstance(value, (int, float))
        and math.isfinite(float(value))
    ]

    if not values:
        return None

    return round(float(pd.Series(values).median()), 2)


summary_rows = []

for sdk in SDK_ORDER:
    label = SDK_LABELS[sdk]
    subset = normalized_df[
        normalized_df["sdk"] == sdk
    ]

    total = len(subset)
    passed = int((subset["status"] == "passed").sum())
    failed = int((subset["status"] == "failed").sum())
    pending = int(
        (subset["status"] == "pending_review").sum()
    )
    runtime_errors = int(
        (subset["status"] == "runtime_error").sum()
    )
    unsupported = int(
        (subset["status"] == "unsupported").sum()
    )

    evaluated = passed + failed

    summary_rows.append({
        "sdk": sdk,
        "SDK": label,
        "total_selected_cases": total,
        "passed": passed,
        "failed": failed,
        "pending_review": pending,
        "runtime_error": runtime_errors,
        "unsupported": unsupported,
        "evaluated_cases": evaluated,
        "pass_rate_evaluated": (
            round(passed / evaluated, 4)
            if evaluated
            else None
        ),
        "runtime_completion_rate": (
            round(
                (
                    total
                    - runtime_errors
                ) / total,
                4,
            )
            if total
            else None
        ),
        "median_latency_s": median_or_none(
            subset["latency_s"]
        ),
        "median_tool_calls": median_or_none(
            subset["tool_call_count"]
        ),
        "median_total_tokens": median_or_none(
            subset["total_tokens"]
        ),
    })

sdk_summary_df = pd.DataFrame(summary_rows)

display(sdk_summary_df)

,sdk,SDK,total_selected_cases,passed,failed,pending_review,runtime_error,unsupported,evaluated_cases,pass_rate_evaluated,runtime_completion_rate,median_latency_s,median_tool_calls,median_total_tokens
0,deepagents,Deep Agents SDK,17,14,1,1,1,0,15,0.9333,0.9412,8.31,2.0,7271.0
1,openai_agents,OpenAI Agents SDK,18,13,1,4,0,0,14,0.9286,1.0000,6.15,2.0,2256.0
2,pi_agent,Pi Agent SDK,18,14,0,2,0,2,14,1.0000,1.0000,NaN,1.5,1625.0


# 07 Each cases comparison (Overview)

In [27]:
common_case_rows = []

for case_id in EXPECTED_CASES:
    case_group = normalized_df[
        normalized_df["case_id"] == case_id
    ]

    statuses = {
        row["sdk"]: row["status"]
        for _, row in case_group.iterrows()
    }

    all_present = all(
        sdk in statuses
        for sdk in SDK_ORDER
    )

    all_evaluable = all(
        statuses.get(sdk) in {"passed", "failed"}
        for sdk in SDK_ORDER
    )

    common_case_rows.append({
        "case_id": case_id,
        # "all_sdks_present": all_present,
        # "all_sdks_evaluable": all_evaluable,
        **{
            SDK_LABELS[sdk]: STATUS_LABELS.get(
                statuses.get(sdk),
                "MISSING",
            )
            for sdk in SDK_ORDER
        },
    })

common_cases_df = pd.DataFrame(common_case_rows)

display(common_cases_df)

,case_id,Deep Agents SDK,OpenAI Agents SDK,Pi Agent SDK
0,math_001,PASS,PASS,PASS
1,math_002,PASS,PASS,PASS
2,math_003,FAIL,FAIL,PASS
3,mcp_001,PASS,PASS,PASS
4,mcp_002,PASS,PASS,PASS
5,mcp_003,PASS,PASS,PASS
6,web_native_001,PENDING,PENDING,UNSUPPORTED
7,web_native_002,RUNTIME_ERROR,PENDING,UNSUPPORTED
8,planning_001,PASS,PENDING,PENDING
9,planning_002,PASS,PENDING,PENDING


# 08 Each cases comparison (Only cases that we can compare)

In [25]:
common_evaluable_case_ids = common_cases_df.loc[
    common_cases_df["all_sdks_evaluable"],
    "case_id",
].tolist()

strict_rows = []

for sdk in SDK_ORDER:
    label = SDK_LABELS[sdk]

    subset = normalized_df[
        (normalized_df["sdk"] == sdk)
        & normalized_df["case_id"].isin(
            common_evaluable_case_ids
        )
    ]

    passed = int(
        (subset["status"] == "passed").sum()
    )

    total = len(subset)

    strict_rows.append({
        "sdk": sdk,
        "SDK": label,
        "common_evaluable_cases": total,
        "passed": passed,
        "failed": total - passed,
        "strict_pass_rate": (
            round(passed / total, 4)
            if total
            else None
        ),
    })

strict_summary_df = pd.DataFrame(strict_rows)

print(
    "Common evaluable cases:",
    len(common_evaluable_case_ids),
)

display(strict_summary_df)

Common evaluable cases: 13


,sdk,SDK,common_evaluable_cases,passed,failed,strict_pass_rate
0,deepagents,Deep Agents SDK,13,12,1,0.9231
1,openai_agents,OpenAI Agents SDK,13,12,1,0.9231
2,pi_agent,Pi Agent SDK,13,13,0,1.0000


In [26]:
common_evaluable_case_ids = common_cases_df.loc[
    common_cases_df["all_sdks_evaluable"],
    "case_id",
].tolist()

strict_rows = []

for sdk in SDK_ORDER:
    label = SDK_LABELS[sdk]

    subset = normalized_df[
        (normalized_df["sdk"] == sdk)
        & normalized_df["case_id"].isin(
            common_evaluable_case_ids
        )
    ]

    passed = int(
        (subset["status"] == "passed").sum()
    )

    total = len(subset)

    strict_rows.append({
        "sdk": sdk,
        "SDK": label,
        "common_evaluable_cases": total,
        "passed": passed,
        "failed": total - passed,
        "strict_pass_rate": (
            round(passed / total, 4)
            if total
            else None
        ),
    })

strict_summary_df = pd.DataFrame(strict_rows)

print(
    "Common evaluable cases:",
    len(common_evaluable_case_ids),
)

display(strict_summary_df)

Common evaluable cases: 13


,sdk,SDK,common_evaluable_cases,passed,failed,strict_pass_rate
0,deepagents,Deep Agents SDK,13,12,1,0.9231
1,openai_agents,OpenAI Agents SDK,13,12,1,0.9231
2,pi_agent,Pi Agent SDK,13,13,0,1.0000
